In [1]:
import pandas as pd
import numpy as np
import mysql.connector as sql

In [2]:
file_path = "Luxury_Housing_Bangalore.csv"
df = pd.read_csv(file_path)

print("Original shape:", df.shape)
print("Original columns:", df.columns.tolist())

Original shape: (101000, 18)
Original columns: ['Property_ID', 'Micro_Market', 'Project_Name', 'Developer_Name', 'Unit_Size_Sqft', 'Configuration', 'Ticket_Price_Cr', 'Transaction_Type', 'Buyer_Type', 'Purchase_Quarter', 'Connectivity_Score', 'Amenity_Score', 'Possession_Status', 'Sales_Channel', 'NRI_Buyer', 'Locality_Infra_Score', 'Avg_Traffic_Time_Min', 'Buyer_Comments']


In [3]:
df.columns = df.columns.str.strip().str.lower()

In [4]:
# rename only what we need
df.rename(columns={
    "property_id": "project_id",
    "developer_name": "builder"
}, inplace=True)

print("Cleaned columns:", df.columns.tolist())
df.drop_duplicates(inplace=True)

Cleaned columns: ['project_id', 'micro_market', 'project_name', 'builder', 'unit_size_sqft', 'configuration', 'ticket_price_cr', 'transaction_type', 'buyer_type', 'purchase_quarter', 'connectivity_score', 'amenity_score', 'possession_status', 'sales_channel', 'nri_buyer', 'locality_infra_score', 'avg_traffic_time_min', 'buyer_comments']


In [5]:
df["amenity_score"] = pd.to_numeric(df["amenity_score"], errors="coerce")
df["amenity_score"] = df["amenity_score"].fillna(df["amenity_score"].median())

df["buyer_type"] = df["buyer_type"].fillna("Unknown")
df["buyer_comments"] = df["buyer_comments"].fillna("No Comments")

df["ticket_price_cr"] = df["ticket_price_cr"].astype(str).str.replace(r"[^\d.]", "", regex=True)
df["ticket_price_cr"] = pd.to_numeric(df["ticket_price_cr"], errors="coerce")

numeric_cols = [
    "unit_size_sqft",
    "connectivity_score",
    "locality_infra_score",
    "avg_traffic_time_min"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -------------------------
# 6. TEXT CLEANING
# -------------------------
title_cols = [
    "micro_market",
    "project_name",
    "builder",
    "transaction_type",
    "buyer_type",
    "possession_status",
    "sales_channel"
]


In [6]:
for col in title_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

df["configuration"] = df["configuration"].astype(str).str.strip().str.upper()
df["nri_buyer"] = df["nri_buyer"].astype(str).str.strip().str.lower()

# -------------------------

In [7]:
df["purchase_quarter"] = pd.to_datetime(df["purchase_quarter"], errors="coerce")
df["purchase_year"] = df["purchase_quarter"].dt.year
df["quarter_number"] = df["purchase_quarter"].dt.quarter

df["unit_size_sqft"] = df["unit_size_sqft"].replace(0, np.nan)
df["price_per_sqft"] = (df["ticket_price_cr"] * 10000000) / df["unit_size_sqft"]

positive_keywords = [
    "loved",
    "excellent",
    "great value",
    "underpriced",
    "will buy",
    "interested",
    "book",
    "confirmed"
]

df["booking_flag"] = df["buyer_comments"].astype(str).str.lower().apply(
    lambda x: 1 if any(word in x for word in positive_keywords) else 0
)

In [8]:
df["nri_buyer_flag"] = df["nri_buyer"].map({"yes": 1, "no": 0}).fillna(0).astype(int)

# -------------------------
# 8. SAVE CLEANED CSV
# -------------------------
df.to_csv("cleaned_data.csv", index=False)
print("✅ Cleaning completed successfully")

# -------------------------
# 9. PREPARE FOR SQL
# -------------------------
df = df.replace({np.nan: None})

✅ Cleaning completed successfully


In [9]:
required_cols = [
    "project_id",
    "micro_market",
    "project_name",
    "builder",
    "unit_size_sqft",
    "configuration",
    "ticket_price_cr",
    "transaction_type",
    "buyer_type",
    "purchase_quarter",
    "purchase_year",
    "quarter_number",
    "connectivity_score",
    "amenity_score",
    "possession_status",
    "sales_channel",
    "nri_buyer",
    "nri_buyer_flag",
    "locality_infra_score",
    "avg_traffic_time_min",
    "buyer_comments",
    "booking_flag",
    "price_per_sqft"
]

In [10]:
print("Checking required columns...")
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in dataframe: {missing_cols}")

data = df[required_cols].values.tolist()
print("Rows ready for insert:", len(data))
print("Columns ready for insert:", len(required_cols))

Checking required columns...
Rows ready for insert: 100000
Columns ready for insert: 23


In [11]:
my_db = sql.connect(
    host="localhost",
    user="root",
    password="12345678"
)

cursor = my_db.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS real_estate_db")
cursor.close()
my_db.close()
print("✅ Database checked/created")

✅ Database checked/created


In [12]:
my_db = sql.connect(
    host="localhost",
    user="root",
    password="12345678",
    database="real_estate_db"
)

cursor = my_db.cursor()

# -------------------------
# 12. DROP OLD TABLE
# -------------------------
cursor.execute("DROP TABLE IF EXISTS luxury_housing")
my_db.commit()

In [13]:
create_query = """
CREATE TABLE luxury_housing (
    project_id VARCHAR(50),
    micro_market VARCHAR(100),
    project_name VARCHAR(100),
    builder VARCHAR(100),
    unit_size_sqft FLOAT,
    configuration VARCHAR(50),
    ticket_price_cr FLOAT,
    transaction_type VARCHAR(50),
    buyer_type VARCHAR(50),
    purchase_quarter DATE,
    purchase_year INT,
    quarter_number INT,
    connectivity_score FLOAT,
    amenity_score FLOAT,
    possession_status VARCHAR(50),
    sales_channel VARCHAR(50),
    nri_buyer VARCHAR(10),
    nri_buyer_flag INT,
    locality_infra_score FLOAT,
    avg_traffic_time_min FLOAT,
    buyer_comments TEXT,
    booking_flag INT,
    price_per_sqft FLOAT
)
"""

cursor.execute(create_query)
my_db.commit()
print("✅ Table created successfully")

✅ Table created successfully


In [14]:
insert_query = """
INSERT INTO luxury_housing (
    project_id,
    micro_market,
    project_name,
    builder,
    unit_size_sqft,
    configuration,
    ticket_price_cr,
    transaction_type,
    buyer_type,
    purchase_quarter,
    purchase_year,
    quarter_number,
    connectivity_score,
    amenity_score,
    possession_status,
    sales_channel,
    nri_buyer,
    nri_buyer_flag,
    locality_infra_score,
    avg_traffic_time_min,
    buyer_comments,
    booking_flag,
    price_per_sqft
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

batch_size = 5000

for i in range(0, len(data), batch_size):
    batch = data[i:i + batch_size]
    cursor.executemany(insert_query, batch)
    my_db.commit()
    print(f"✅ Inserted rows: {i + len(batch)}")

cursor.execute("SELECT COUNT(*) FROM luxury_housing")
print("Total rows in table:", cursor.fetchone()[0])

# ==================================
# SQL VALIDATION QUERIES
# ==================================

cursor.execute("""
SELECT booking_flag, COUNT(*) AS count
FROM luxury_housing
GROUP BY booking_flag
""")

print("\nBooking Flag Count:")
for row in cursor.fetchall():
    print(row)


cursor.execute("""
SELECT builder,
AVG(ticket_price_cr) AS avg_price
FROM luxury_housing
GROUP BY builder
ORDER BY avg_price DESC
LIMIT 10
""")

print("\nTop 10 Builders by Average Ticket Price:")
for row in cursor.fetchall():
    print(row)


cursor.close()
my_db.close()

✅ Inserted rows: 5000
✅ Inserted rows: 10000
✅ Inserted rows: 15000
✅ Inserted rows: 20000
✅ Inserted rows: 25000
✅ Inserted rows: 30000
✅ Inserted rows: 35000
✅ Inserted rows: 40000
✅ Inserted rows: 45000
✅ Inserted rows: 50000
✅ Inserted rows: 55000
✅ Inserted rows: 60000
✅ Inserted rows: 65000
✅ Inserted rows: 70000
✅ Inserted rows: 75000
✅ Inserted rows: 80000
✅ Inserted rows: 85000
✅ Inserted rows: 90000
✅ Inserted rows: 95000
✅ Inserted rows: 100000
Total rows in table: 100000

Booking Flag Count:
(1, 45349)
(0, 54651)

Top 10 Builders by Average Ticket Price:
('Sobha', 12.887277791592819)
('Total Environment', 12.826205674481079)
('L&T Realty', 12.789966443091123)
('Godrej', 12.766584827509618)
('Puravankara', 12.735030247653189)
('Rmz', 12.729549519474043)
('Prestige', 12.721969587603015)
('Tata Housing', 12.660382572390107)
('Snn Raj', 12.61332724261465)
('Embassy', 12.59307051305654)


In [15]:
print("\n✅ ETL Pipeline completed successfully")


✅ ETL Pipeline completed successfully
